In [ ]:
import json
import re
import os

OUTPUT_DIR = "../outputs"

# 앞에서 저장한 OCR 결과 불러오기
with open(os.path.join(OUTPUT_DIR, "ocr_sample.json"), "r", encoding="utf-8") as f:
    results = json.load(f)

print(f"불러온 페이지 수: {len(results)}")
print(f"첫 페이지 미리보기: {results[0]['text'][:100]}")

불러온 페이지 수: 5
첫 페이지 미리보기: • 경제생활에서기본연산의활용을이해한다.
• 경제생활에서수단위의활용을이해한다.
• 경제지표의의미와종류를이해한다.

• 환율의의미와환율계산법을이해한다.

• 훈세금의종류와MS계산법을이


In [ ]:
def classify_block(text):
    text = text.strip()
    
    # 페이지 번호 패턴 - 가장 먼저 체크
    if re.search(r'\d+\s*(ee|ㅎ|ㅇ|｜|\])\s*\S*\s*(수와경제생활|경제생활)', text):
        return "page_number"
    if re.search(r'^\d+\s*[ㅎㅇ｜]\s*\S+', text):
        return "page_number"
    
    # 학습목표 패턴
    if text.startswith('•') and '이해한다' in text:
        return "learning_goal"
    
    # 수식 패턴 - 제목보다 먼저 체크
    if re.search(r'=\s*\d|=\s*[A-Z]|\d\s*[+\-×÷]\s*\d', text):
        return "formula"
    if re.search(r'GNP|GDP|GNI', text):
        return "formula"
    
    # 제목 패턴
    if len(text) < 25 and re.match(r'^\d+', text) and not re.search(r'[=+\-×÷]', text):
        return "title"
    
    # 표 패턴
    if re.search(r'단위\s*:', text) or re.search(r'\d+\s*조원|\d+\s*%', text):
        return "table"
    
    return "body"

In [ ]:
structured = [structure_page(r) for r in results]

for page in structured:
    print(f"\n{'='*40}")
    print(f"[{page['page_id']}페이지]")
    for b in page["blocks"]:
        print(f"  [{b['type']:12s}] {b['content'][:50]}")


[10페이지]
  [learning_goal] • 경제생활에서기본연산의활용을이해한다.
  [learning_goal] • 경제생활에서수단위의활용을이해한다.
  [learning_goal] • 경제지표의의미와종류를이해한다.
  [learning_goal] • 환율의의미와환율계산법을이해한다.
  [learning_goal] • 훈세금의종류와MS계산법을이해한다.

[11페이지]
  [title       ] 2016년BF예산어디에쓰이나
  [body        ] 분야별재원배분 ※ ( ) 는전년대비증감률
  [table       ] ( 단위 : 조원 )
  [body        ] 자료 : 기획재정부
  [body        ] > 분야별예산배분
  [body        ] LED다음과He수와기호를가지고있다고하자.
  [body        ] • , 1, 2, 3, 4, 5, 6, 7, 8, 9
  [body        ] ( ), +, -, × 훈층 ’ =
  [body        ] 아래와같은활동으로연산에대한HSS해보자.
  [body        ] (1) 두개의연산만을사용하여73을만들어Sct.
  [body        ] (2) 사칙연산을모두사용하여두자리의짝수를만들어본다,
  [body        ] GED다양한경우가나올 + 있으며연산에대한연습을한다.
  [formula     ] (1) 100 — 30 + 3 = 73, (5x3) ㅜ + 58 = 73, (75 + 5)
  [formula     ] (2) {(10x12) + (4x8)} + 34 — 26 = 18, {(17 + 3) x 
  [title       ] 1. 경제지포 ㅇ ㅎ 15

[12페이지]
  [body        ] 한교실에여학생과남학생의비율은1 :2라고한다.
  [body        ] 교실의총학생수가30명일때, 남학생의수는몇Bola] 생각해보아라.
  [body        ] 어떤현상을수학적으로표현할때종종비율이라는HSWAS사용한다. 비율의수학적의미는
  [

In [ ]:
output_path = os.path.join(OUTPUT_DIR, "structured_sample.json")
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(structured, f, ensure_ascii=False, indent=2)

print(f"저장 완료: {output_path}")

# 요약 출력
for page in structured:
    types = [b["type"] for b in page["blocks"]]
    from collections import Counter
    count = Counter(types)
    print(f"[{page['page_id']}페이지] {dict(count)}")

저장 완료: ../outputs\structured_sample.json
[10페이지] {'learning_goal': 5}
[11페이지] {'title': 2, 'body': 10, 'table': 1, 'formula': 2}
[12페이지] {'body': 19, 'page_number': 1}
[13페이지] {'body': 11, 'formula': 13, 'title': 1}
[14페이지] {'formula': 5, 'body': 17, 'table': 1, 'page_number': 1}
